In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc
cc.setupEnvironment()

In [2]:
spark = cc.startLocalCluster("DIM_WEATHER",4)
spark.getActiveSession()

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.appName("dim_weather").getOrCreate()

columns = ["weather_id", "condition"]

weather_types = "../weathertypes.csv"

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(weather_types)
df.write.format("delta").mode("overwrite").saveAsTable("dim_weather")

cc.set_connectionProfile("velo_db")
print(cc.create_jdbc())
df.write \
    .format("jdbc") \
    .option("driver", "org.postgresql.Driver") \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "dimWeather") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "weather_id") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1001) \
    .mode("overwrite") \
    .save()
# Create DataFrame
# Show results
df.show()


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/C:/kdg/data4/shared/sharedbicycleproject-team46/sharedBicycleProject_team46/DimTables/weathertypes.csv.